# Lesson 02 - A Microsoft Agent Framework felfedezése

A **Microsoft Agent Framework (MAF)** egy egységes keretrendszer AI ügynökök építéséhez. Tiszta, összetett architektúrát biztosít négy alapvető építőelemmel:

- **Client** – csatlakozik egy AI modell végponthoz és kezeli a kommunikációt
- **Agent** – becsomagolja az ügyfelet utasításokkal és eszközdefiníciókkal
- **Tools** – kibővítik az ügynök képességeit egyedi függvényekkel, amelyeket a modell meghívhat
- **Session** – fenntartja a beszélgetés előzményeit többfordulós interakciókhoz

Ebben a leckében egy **utazásfoglaló ügynököt** építünk, amely ezen koncepciók segítségével ellenőrzi a célállomás elérhetőségét.


## Beállítás


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Az Agent Framework architektúrájának megértése

A Microsoft Agent Framework rétegzett architektúrát követ:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Ügyfél** – Egy `AzureAIProjectAgentProvider` kapcsolódik egy Azure OpenAI telepítéshez. Kezeli a hitelesítést, a kérésformázást és a válaszok elemzését.
2. **Agent** – Az ügyféltől a `provider.create_agent()` segítségével létrehozott agent kombinálja a modellhozzáférést az utasításokkal (rendszerprompt) és az eszközökkel.
3. **Eszközök** – Python függvények, amelyeket `@tool` díszítővel láttak el, és amelyeket az agent meghívhat műveletek végrehajtására vagy adatok lekérésére.
4. **Munkamenet** – Egy `AgentSession` objektum (az `agent.create_session()` segítségével létrehozva), amely tárolja a beszélgetés előzményeit, lehetővé téve a többfordulós párbeszédet, ahol az agent emlékszik az előző kontextusra.

Építsük fel rétegről rétegre.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Eszközök hozzáadása az @tool dekorátorral

Az eszközök lehetővé teszik, hogy az ügynökök a szöveg generáláson túl intézkedéseket hajtsanak végre. Az `@tool` dekorátor egy normál Python függvényt átalakít valami olyanná, amit az ügynök hívhat.

Főbb pontok:
- Használd az `Annotated[type, "description"]` formátumot, hogy a modell megértse az egyes paramétereket.
- A docstring lesz az eszköz leírása, amit a modell lát.
- Az `approval_mode="never_require"` azt jelenti, hogy az eszköz automatikusan fut, felhasználói megerősítés nélkül.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Ügynök létrehozása eszközökkel

Most összevonjuk az ügyfelet, az utasításokat és az eszközöket egy ügynökké. Az `instructions` szolgálnak rendszerutasításként — meghatározzák az ügynök személyiségét és viselkedését.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Többfordulós beszélgetések munkamenetekkel

Egy `AgentSession` (amit az `agent.create_session()` hoz létre) nyomon követi a beszélgetés összes üzenetét. Ha ugyanazt a munkamenetet adjuk át minden `agent.run()` hívásnak, az ügynök hozzáfér a teljes beszélgetési előzményhez, és visszautalhat korábbi üzenetekre.

A `tools=[check_destination_availability]` átadásával az ügynök meghívhatja az elérhetőség-ellenőrzőnket minden fordulóban.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Összefoglalás

Ebben a leckében felfedezted a Microsoft Agent Framework négy pillérét:

| Fogalom | Amit megtanultál |
|---------|------------------|
| **Ügyfél** | Az `AzureAIProjectAgentProvider` hitelesítésen alapuló authentikációval csatlakozik az Azure OpenAI-hoz |
| **Agent** | A `provider.create_agent()` egy modellt csomagol össze utasításokkal és névvel |
| **Eszközök** | A `@tool` dekorátor lehetővé teszi Python függvények kitenyésztését az agent számára hívásra |
| **Munkamenet** | Az `agent.create_session()` több körös beszélgetés történetét tartja meg |

Ezek az építőkövek együtt alkotnak olyan agenteket, amelyek tudnak természetes beszélgetést folytatni, külső függvényeket meghívni és kontextust fenntartani – ez alapot ad a későbbi leckék haladó agentikus mintáihoz.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Figyelmeztetés**:
Ezt a dokumentumot az AI fordító szolgáltatás [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével fordítottuk. Bár igyekszünk a pontosságra, kérjük, vegye figyelembe, hogy az automatikus fordítások tartalmazhatnak hibákat vagy pontatlanságokat. Az eredeti dokumentum anyanyelvű változata tekintendő hivatalos forrásnak. Fontos információk esetén szakmai emberi fordítást javaslunk. Nem vállalunk felelősséget az e fordítás használatából eredő félreértésekért vagy téves értelmezésekért.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
